# 🌍 Humanitarian Data Explorer — Guided Notebook
Welcome! This notebook walks through the same analysis as `src/analyze.py`, step by step, so you can see and change every decision. No data-science background required — each cell explains **why** before showing **how**.

**Before running:** execute `python src/fetch_data.py` in a terminal so the `data/` folder is populated.

## 1. Load the cleaned country snapshot
Why: one row per sovereign country with the four factors already averaged over 2015–2024. Reading it here avoids re-downloading anything.

In [ ]:
import pandas as pd
pd.set_option('display.width', 160)
snap = pd.read_csv('../data/country_snapshot.csv')
print(snap.shape, 'countries x columns')
snap.head()

## 2. The four factors at a glance
| Column | Factor | Higher means… |
|---|---|---|
| `hazard_events_per_year` | natural-hazard exposure (EM-DAT via UN HDX) | more disaster triggers |
| `refugees_per_1000` | forced displacement burden (UNHCR Global Trends) | more people hosted per resident |
| `gdp_per_capita` | economic capacity (World Bank) | more ability to cope |
| `under5_mortality`, `undernourishment` | health vulnerability (World Bank) | more susceptibility |

Why per-capita rates? Absolute counts mostly rank countries by population size — Lebanon's refugee *burden* is invisible in totals but obvious per 1,000 residents.

In [ ]:
snap[['country','hazard_events_per_year','refugees_per_1000',
      'gdp_per_capita','under5_mortality']].describe().T.round(1)

## 3. Wealth vs. child survival — the central inequality story
Why log scales? Both variables span several orders of magnitude; on linear axes all poor countries pile into one corner. Log-log turns the relationship into an almost straight line students can read.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
d = snap.dropna(subset=['gdp_per_capita','under5_mortality'])
plt.figure(figsize=(9,5.5))
plt.scatter(d['gdp_per_capita'], d['under5_mortality'], alpha=.7, edgecolor='w')
plt.xscale('log'); plt.yscale('log')
b,a = np.polyfit(np.log10(d.gdp_per_capita), np.log10(d.under5_mortality), 1)
xs = np.logspace(np.log10(d.gdp_per_capita.min()), np.log10(d.gdp_per_capita.max()))
plt.plot(xs, 10**(a+b*np.log10(xs)), '--r', label=f'log-log fit (slope {b:.2f})')
for iso in ['AFG','SOM','TCD','JPN','NOR','BRA','IND','NGA']:
    r = d[d.iso3==iso]
    if len(r): plt.annotate(r.country.iloc[0], (r.gdp_per_capita.iloc[0], r.under5_mortality.iloc[0]), fontsize=8)
plt.xlabel('GDP per capita (const. 2015 US$, log)'); plt.ylabel('Under-5 deaths per 1,000 births (log)')
plt.title('Poorer countries lose more children — even after accounting for wealth')
plt.legend(); plt.tight_layout(); plt.show()
print('Pearson r raw     :', round(d.gdp_per_capita.corr(d.under5_mortality),2))
print('Pearson r log-log :', round(np.log10(d.gdp_per_capita).corr(np.log10(d.under5_mortality)),2))

## 4. Who hosts refugees relative to their own size?
Why this ranking matters: it shows the displacement burden falls mostly on relatively poor neighbouring countries, not on wealthy states — a key equity finding for humanitarian advocacy.

In [ ]:
snap.nlargest(10,'refugees_per_1000')[['country','refugees_per_1000','gdp_per_capita']].round(1)

## 5. Composite vulnerability score
Why combine factors? Crises bite hardest where several problems overlap. We percentile-rank each factor (1 = worst), average them, and plot the top 15. Next step for students: try PCA weights instead of equal ones.

In [ ]:
cols = ['hazard_events_per_year','under5_mortality','undernourishment','refugees_per_1000']
r = snap.copy()
for c in cols: r[c] = r[c].rank(pct=True, ascending=False)
r['burden'] = r[cols].mean(axis=1)
top = r.nsmallest(15,'burden')
top.plot.barh(y='burden', x='country', figsize=(8,6), legend=False, color='#d6604d')
plt.gca().invert_yaxis(); plt.xlabel('Composite burden score (longer = more vulnerable)')
plt.title('Where humanitarian factors overlap (2015–2024)'); plt.tight_layout(); plt.show()
top[['country','burden']].round(3)

## 6. Discussion prompts for class
1. Figure 1 (in `../figures/`) shows rising disaster counts since 2000 — how much is real climate change vs. better reporting? What data would distinguish them?
2. The correlation in §3 is strong but not causal. Design a study that could test whether *income* causes lower child mortality.
3. Rebuild the burden score dropping `refugees_per_1000`. Which countries enter/leave the top 15, and what does that tell you about index construction choices?
4. Pick your country on the map (`fig2_map_hazard.png`). Do its four factor values match what you know about it? Where might the data be wrong?